In [10]:
# ── Imports & Configuration ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os, glob, warnings
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW  = os.path.join(BASE, "data", "raw")
SEASON_TAG  = "serie_a_2025_2026"
SEASON_LABEL = "2025_2026"

B2B_POSITIONS = {"MC"}
MIN_MINUTES  = 900

PILLAR_WEIGHTS = {
    "Build-up & Circulation": 0.30,
    "Switching Play":         0.20,
    "Progressive Passing":    0.30,
    "Defensive Contribution": 0.20,
}

KPI_MAP = {
    "Build-up & Circulation": {
        "total_passes_p90":          {"weight": 0.25, "negative": False},
        "pass_completion_pct":       {"weight": 0.25, "negative": False},
        "backward_lateral_pct":      {"weight": 0.20, "negative": False},
        "forward_pass_completion_pct":{"weight": 0.30, "negative": False},
    },
    "Switching Play": {
        "switches_p90":              {"weight": 0.40, "negative": False},
        "long_ball_completion_pct":  {"weight": 0.30, "negative": False},
        "long_balls_p90":            {"weight": 0.30, "negative": False},
    },
    "Progressive Passing": {
        "forward_passes_p90":        {"weight": 0.20, "negative": False},
        "final_third_passes_p90":    {"weight": 0.15, "negative": False},
        "through_balls_p90":         {"weight": 0.15, "negative": False},
        "progressive_passes_p90":    {"weight": 0.20, "negative": False},
        "progressive_carries_p90":   {"weight": 0.15, "negative": False},
        "key_passes_p90":            {"weight": 0.15, "negative": False},
    },
    "Defensive Contribution": {
        "ball_recoveries_p90":       {"weight": 0.20, "negative": False},
        "tackles_won_p90":           {"weight": 0.20, "negative": False},
        "interceptions_p90":         {"weight": 0.20, "negative": False},
        "duels_won_pct":             {"weight": 0.20, "negative": False},
        "possession_won_p90":        {"weight": 0.20, "negative": False},
    },
}

KPI_LABELS = {
    "total_passes_p90":           "Total Passes",
    "pass_completion_pct":        "Pass Completion %",
    "backward_lateral_pct":       "Back/Lateral Pass %",
    "forward_pass_completion_pct":"Forward Pass Compl. %",
    "switches_p90":               "Switches of Play",
    "long_ball_completion_pct":   "Long Ball Compl. %",
    "long_balls_p90":             "Long Balls",
    "forward_passes_p90":         "Forward Passes",
    "final_third_passes_p90":     "Final Third Passes",
    "through_balls_p90":          "Through Balls",
    "progressive_passes_p90":     "Progressive Passes",
    "progressive_carries_p90":    "Progressive Carries",
    "key_passes_p90":             "Key Passes",
    "ball_recoveries_p90":        "Ball Recoveries",
    "tackles_won_p90":            "Tackles Won",
    "interceptions_p90":          "Interceptions",
    "duels_won_pct":              "Duels Won %",
    "possession_won_p90":         "Possession Won",
}

print(f"Target season: {SEASON_TAG}")
print(f"Events dir exists: {os.path.isdir(os.path.join(RAW, SEASON_TAG, 'events'))}")
n_files = len(glob.glob(os.path.join(RAW, SEASON_TAG, "events", "*.csv")))
print(f"Match files: {n_files}")
print("Setup complete ✓")

Target season: serie_a_2025_2026
Events dir exists: True
Match files: 319
Setup complete ✓


In [11]:
# ── Load 2025/26 events ──────────────────────────────────────────────────────
evt_dir = os.path.join(RAW, SEASON_TAG, "events")
files = sorted(glob.glob(os.path.join(evt_dir, "*.csv")))
print(f"Loading {len(files)} match files...")

frames = []
for f in files:
    try:
        df = pd.read_csv(f, low_memory=False)
        df["season"] = SEASON_LABEL
        df["gameweek"] = os.path.basename(f).split("_")[0]
        frames.append(df)
    except:
        continue

all_events = pd.concat(frames, ignore_index=True)
b2b_events = all_events[all_events["position"].isin(B2B_POSITIONS)].copy()

print(f"Total events: {len(all_events):,}")
print(f"Box-to-Box MF events: {len(b2b_events):,}")
print(f"Unique B2B MF players: {b2b_events['player_name'].nunique()}")

Loading 319 match files...
Total events: 532,428
Box-to-Box MF events: 82,820
Unique B2B MF players: 189
Total events: 532,428
Box-to-Box MF events: 82,820
Unique B2B MF players: 189


In [12]:
# ── Estimate minutes & compute raw KPIs ──────────────────────────────────────

# --- Minutes ---
dm = b2b_events.copy()
dm["time_min"] = pd.to_numeric(dm["time_min"], errors="coerce")

player_match = (
    dm.groupby(["player_name", "player_id", "team_name", "season", "match_id"])
    ["time_min"].max().reset_index().rename(columns={"time_min": "last_event_min"})
)
minutes_df = (
    player_match.groupby(["player_name", "player_id", "team_name", "season"])
    .agg(matches=("match_id", "nunique"), total_minutes=("last_event_min", "sum"))
    .reset_index()
)

# --- Raw KPIs ---
df = b2b_events.copy()
for c in ["type_id", "outcome", "x", "y"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["Pass End X"] = pd.to_numeric(df["Pass End X"], errors="coerce")
df["Pass End Y"] = pd.to_numeric(df["Pass End Y"], errors="coerce")

grp = ["player_name", "player_id", "team_name", "season"]

def _cnt(mask):
    return df[mask].groupby(grp).size()

kpis = pd.DataFrame()

# ── Build-up & Circulation ──
kpis["total_passes"]      = _cnt(df["type_id"] == 1)
kpis["passes_completed"]  = _cnt((df["type_id"] == 1) & (df["outcome"] == 1))
kpis["backward_lateral"]  = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) & (df["Pass End X"] <= df["x"])
)

# ── Switching Play ──
kpis["switches"] = _cnt(
    (df["type_id"] == 1) & (df["Switch of play"] != "N/A") & df["Switch of play"].notna()
)
kpis["long_balls_total"] = _cnt(
    (df["type_id"] == 1) & (df["Long ball"] != "N/A") & df["Long ball"].notna()
)
kpis["long_balls_completed"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) & (df["Long ball"] != "N/A") & df["Long ball"].notna()
)

# ── Progressive Passing ──
# Forward passes (completed): Pass End X > x
kpis["forward_passes"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) & (df["Pass End X"] > df["x"])
)
# Forward passes attempted (for completion %)
kpis["forward_passes_attempted"] = _cnt(
    (df["type_id"] == 1) & (df["Pass End X"] > df["x"])
)
# Final third passes
kpis["final_third_passes"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) & (df["Pass End X"] > 66.7)
)
# Through balls
kpis["through_balls"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) & (df["Through ball"] != "N/A") & df["Through ball"].notna()
)
# Progressive passes: completed passes that advance ball ≥ 10 units toward goal
kpis["progressive_passes"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) & ((df["Pass End X"] - df["x"]) >= 10)
)
# Key passes: passes with Assist qualifier (Q210)
kpis["key_passes"] = _cnt(
    (df["type_id"] == 1) & (df["Assist"] != "N/A") & df["Assist"].notna()
)
# Progressive carries: successful take-ons (type 3, outcome 1) that advance ball ≥ 10 units
kpis["progressive_carries"] = _cnt(
    (df["type_id"] == 3) & (df["outcome"] == 1) & ((df["Pass End X"] - df["x"]).fillna(0) >= 10)
)
# Fallback: if no end-x available for take-ons, use all successful take-ons as proxy
if kpis["progressive_carries"].sum() == 0:
    kpis["progressive_carries"] = _cnt((df["type_id"] == 3) & (df["outcome"] == 1))

# ── Defensive Contribution ──
kpis["ball_recoveries"] = _cnt(df["type_id"] == 49)
kpis["tackles_won"]     = _cnt((df["type_id"] == 7) & (df["outcome"] == 1))
kpis["tackles_total"]   = _cnt(df["type_id"] == 7)
kpis["interceptions"]   = _cnt(df["type_id"] == 8)
# Aerial duels
kpis["aerials_won"]     = _cnt((df["type_id"] == 44) & (df["outcome"] == 1))
kpis["aerials_total"]   = _cnt(df["type_id"] == 44)
# Take-ons (offensive duels)
kpis["take_ons_won"]    = _cnt((df["type_id"] == 3) & (df["outcome"] == 1))
kpis["take_ons_total"]  = _cnt(df["type_id"] == 3)

kpis = kpis.fillna(0).reset_index()

# --- Merge & per-90 ---
merged = kpis.merge(minutes_df[["player_name","player_id","season","matches","total_minutes"]],
                     on=["player_name","player_id","season"], how="left")
player_stats = merged[merged["total_minutes"] >= MIN_MINUTES].copy()

per90_cols = ["total_passes","switches","long_balls_total","long_balls_completed",
              "forward_passes","final_third_passes","through_balls",
              "progressive_passes","progressive_carries","key_passes",
              "ball_recoveries","tackles_won","interceptions"]
for c in per90_cols:
    player_stats[f"{c}_p90"] = (player_stats[c] / player_stats["total_minutes"]) * 90

# Alias so KPI_MAP key "long_balls_p90" resolves correctly
player_stats["long_balls_p90"] = player_stats["long_balls_total_p90"]

# Percentage KPIs
player_stats["pass_completion_pct"] = (player_stats["passes_completed"]/player_stats["total_passes"]*100).fillna(0)
player_stats["backward_lateral_pct"] = (player_stats["backward_lateral"]/player_stats["passes_completed"]*100).fillna(0)
player_stats["long_ball_completion_pct"] = (player_stats["long_balls_completed"]/player_stats["long_balls_total"]*100).fillna(0)
player_stats["forward_pass_completion_pct"] = (player_stats["forward_passes"]/player_stats["forward_passes_attempted"]*100).fillna(0)

# Duels won % = (aerials won + tackles won + take-ons won) / (aerials total + tackles total + take-ons total)
duels_won  = player_stats["aerials_won"] + player_stats["tackles_won"] + player_stats["take_ons_won"]
duels_total = player_stats["aerials_total"] + player_stats["tackles_total"] + player_stats["take_ons_total"]
player_stats["duels_won_pct"] = (duels_won / duels_total * 100).fillna(0)

# Possession won p90 = (tackles won + interceptions + take-ons won) per 90
player_stats["possession_won"] = player_stats["tackles_won"] + player_stats["interceptions"] + player_stats["take_ons_won"]
player_stats["possession_won_p90"] = (player_stats["possession_won"] / player_stats["total_minutes"]) * 90

print(f"Players with ≥{MIN_MINUTES} min: {len(player_stats)}")
player_stats.sort_values("progressive_passes_p90", ascending=False)[
    ["player_name","team_name","matches","total_minutes",
     "progressive_passes_p90","progressive_carries_p90","key_passes_p90",
     "duels_won_pct","possession_won_p90","forward_pass_completion_pct"]
].head(10).round(2)

Players with ≥900 min: 61


,player_name,team_name,matches,total_minutes,progressive_passes_p90,progressive_carries_p90,key_passes_p90,duels_won_pct,possession_won_p90,forward_pass_completion_pct
127,M. Locatelli,Juventus FC,24,1995,19.44,0.18,1.17,60.00,3.38,85.05
20,B. Cristante,AS Roma,28,2313,13.04,0.35,0.54,54.40,2.33,76.34
145,M. de Roon,Atalanta Bergamasca Calcio,26,2135,11.13,0.04,0.51,51.33,1.98,78.53
132,M. Pašalić,Atalanta Bergamasca Calcio,20,1622,11.10,0.28,1.39,59.70,1.50,83.16
151,N. Matić,US Sassuolo Calcio,28,2422,9.40,0.33,0.63,59.26,2.04,78.40
146,N. Barella,FC Internazionale Milano,28,2271,9.31,0.83,2.14,46.72,2.06,73.78
126,M. Koné,AS Roma,26,2376,9.05,1.06,0.91,50.68,2.65,87.53
217,Éderson,Atalanta Bergamasca Calcio,23,1874,9.03,0.58,0.96,47.37,2.74,83.15
150,N. Fagioli,ACF Fiorentina,14,1120,8.36,0.80,1.37,64.86,2.25,83.07
175,S. Lobotka,SSC Napoli,20,1792,7.89,0.20,0.65,51.22,1.16,87.95


In [13]:
# ── Possession-Based Normalization ──────────────────────────────────────────────
ae = all_events.copy()
ae["type_id"] = pd.to_numeric(ae["type_id"], errors="coerce")
ae["outcome"] = pd.to_numeric(ae["outcome"], errors="coerce")

team_passes = (
    ae[(ae["type_id"] == 1) & (ae["outcome"] == 1)]
    .groupby(["match_id", "team_name"]).size()
    .reset_index(name="passes")
)

match_teams = team_passes.groupby("match_id")["team_name"].apply(list).reset_index()
match_teams = match_teams[match_teams["team_name"].apply(len) == 2]

poss_rows = []
for _, r in match_teams.iterrows():
    mid = r["match_id"]
    t1, t2 = r["team_name"]
    mp = team_passes[team_passes["match_id"] == mid].set_index("team_name")["passes"]
    total = mp.get(t1, 0) + mp.get(t2, 0)
    if total > 0:
        poss_rows.append({"team_name": t1, "match_id": mid, "poss_pct": mp.get(t1, 0) / total * 100})
        poss_rows.append({"team_name": t2, "match_id": mid, "poss_pct": mp.get(t2, 0) / total * 100})

poss_df = pd.DataFrame(poss_rows)
team_poss = poss_df.groupby("team_name")["poss_pct"].mean().reset_index(name="avg_possession")
league_avg_poss = team_poss["avg_possession"].mean()
team_poss["adj_factor"] = team_poss["avg_possession"] / league_avg_poss

print(f"League-avg possession: {league_avg_poss:.1f}%")

# ── Apply adjustment to count-based p90 metrics ──────────────────────────────
player_stats = player_stats.merge(team_poss[["team_name", "avg_possession", "adj_factor"]], on="team_name", how="left")

count_based_p90 = [
    "total_passes_p90", "switches_p90", "long_balls_total_p90", "long_balls_completed_p90",
    "forward_passes_p90", "final_third_passes_p90", "through_balls_p90",
    "progressive_passes_p90", "progressive_carries_p90", "key_passes_p90",
    "ball_recoveries_p90", "tackles_won_p90", "interceptions_p90",
    "possession_won_p90",
]

for c in count_based_p90:
    player_stats[f"{c}_raw"] = player_stats[c]
    player_stats[c] = player_stats[c] * player_stats["adj_factor"]

# Re-apply alias after adjustment
player_stats["long_balls_p90"] = player_stats["long_balls_total_p90"]

print(f"\nAdjustment applied to {len(count_based_p90)} count-based p90 metrics.")
print("Ratios (pass completion %, long ball compl. %, backward/lateral %, forward pass compl. %, duels won %) NOT adjusted.")
player_stats.sort_values("progressive_passes_p90", ascending=False)[
    ["player_name", "team_name", "avg_possession", "adj_factor", "progressive_passes_p90", "possession_won_p90"]
].head(10).round(2)

League-avg possession: 50.0%

Adjustment applied to 14 count-based p90 metrics.
Ratios (pass completion %, long ball compl. %, backward/lateral %, forward pass compl. %, duels won %) NOT adjusted.


,player_name,team_name,avg_possession,adj_factor,progressive_passes_p90,possession_won_p90
34,M. Locatelli,Juventus FC,58.04,1.16,22.58,3.93
6,B. Cristante,AS Roma,57.81,1.16,15.07,2.70
38,M. de Roon,Atalanta Bergamasca Calcio,56.05,1.12,12.48,2.22
36,M. Pašalić,Atalanta Bergamasca Calcio,56.05,1.12,12.44,1.68
39,N. Barella,FC Internazionale Milano,60.95,1.22,11.35,2.51
33,M. Koné,AS Roma,57.81,1.16,10.47,3.07
60,Éderson,Atalanta Bergamasca Calcio,56.05,1.12,10.12,3.07
52,S. Lobotka,SSC Napoli,60.05,1.20,9.47,1.39
42,N. Fagioli,ACF Fiorentina,52.54,1.05,8.78,2.36
44,N. Pisilli,AS Roma,57.81,1.16,8.62,2.57


In [14]:
# ── Percentiles, Pillar Scores, Composite Score ──────────────────────────────

ps = player_stats.copy()

# ── Percentile ranks ─────────────────────────────────────────
for pillar, kpis_def in KPI_MAP.items():
    for kpi, info in kpis_def.items():
        pc = f"{kpi}_pct"
        ps[pc] = ps[kpi].rank(pct=True) * 100
        if info["negative"]:
            ps[pc] = 100 - ps[pc]

# ── Pillar scores ─────────────────────────────────────────
for pillar, kpis_def in KPI_MAP.items():
    col = f"pillar_{pillar.replace(' ','_').replace('&','and').lower()}"
    ps[col] = sum(ps[f"{k}_pct"] * v["weight"] for k, v in kpis_def.items())

# ── Composite score ──────────────────────────────────────────
ps["composite_score"] = (
    ps["pillar_build-up_and_circulation"] * 0.30 +
    ps["pillar_switching_play"]           * 0.25 +
    ps["pillar_progressive_passing"]      * 0.25 +
    ps["pillar_defensive_contribution"]   * 0.20
)
ps = ps.sort_values("composite_score", ascending=False).reset_index(drop=True)
ps["rank"] = range(1, len(ps)+1)

print(f"Final scored players: {len(ps)}")
ps[["rank","player_name","team_name","matches","total_minutes","composite_score",
    "pillar_build-up_and_circulation","pillar_switching_play",
    "pillar_progressive_passing","pillar_defensive_contribution"]].head(15).round(1)

Final scored players: 61


,rank,player_name,team_name,matches,total_minutes,composite_score,pillar_build-up_and_circulation,pillar_switching_play,pillar_progressive_passing,pillar_defensive_contribution
0,1,M. Locatelli,Juventus FC,24,1995,85.6,73.7,90.5,87.0,95.7
1,2,Éderson,Atalanta Bergamasca Calcio,23,1874,83.6,75.3,94.1,82.0,84.6
2,3,N. Barella,FC Internazionale Milano,28,2271,75.9,61.1,83.8,94.3,65.2
3,4,M. Pašalić,Atalanta Bergamasca Calcio,20,1622,75.6,69.1,81.3,86.7,64.3
4,5,M. Koné,AS Roma,26,2376,75.2,76.1,55.6,88.6,81.3
5,6,N. Fagioli,ACF Fiorentina,14,1120,74.8,72.1,67.4,86.4,73.4
6,7,B. Cristante,AS Roma,28,2313,73.6,51.4,82.3,78.2,90.5
7,8,H. Mkhitaryan,FC Internazionale Milano,21,1713,72.8,63.4,68.2,77.5,86.6
8,9,P. Sučić,FC Internazionale Milano,23,1798,70.9,70.8,79.5,75.1,55.1
9,10,S. Lobotka,SSC Napoli,20,1792,70.8,85.3,71.6,68.4,51.1


## 📊 Interactive Composite Ranking

Bar chart of the top 25 players ranked by composite score, with pillar breakdown on hover.

In [15]:
# ── Interactive stacked bar: Top 25 by composite score ────────────────────────

top_n = ps.head(25).copy().sort_values("composite_score", ascending=True)

pillar_info = [
    ("pillar_build-up_and_circulation",  "Build-up & Circulation (0.30)",  "#1f77b4"),
    ("pillar_switching_play",            "Switching Play (0.20)",          "#ff7f0e"),
    ("pillar_progressive_passing",       "Progressive Passing (0.30)",     "#2ca02c"),
    ("pillar_defensive_contribution",    "Defensive Contribution (0.20)",  "#d62728"),
]

fig_bar = go.Figure()
for col, name, color in pillar_info:
    weight = PILLAR_WEIGHTS[{
        "pillar_build-up_and_circulation": "Build-up & Circulation",
        "pillar_switching_play": "Switching Play",
        "pillar_progressive_passing": "Progressive Passing",
        "pillar_defensive_contribution": "Defensive Contribution",
    }[col]]
    fig_bar.add_trace(go.Bar(
        y=top_n["player_name"] + " (" + top_n["team_name"] + ")",
        x=(top_n[col] * weight).round(1),
        name=name, orientation="h", marker_color=color,
        hovertemplate="%{y}<br>" + name + ": %{x:.1f}<extra></extra>"
    ))

fig_bar.update_layout(
    barmode="stack",
    title="Top 25 Box-to-Box Midfielders — Serie A 2025/26<br><sub>Weighted Composite Score (pillar breakdown)</sub>",
    xaxis_title="Composite Score", yaxis_title="",
    height=700, template="plotly_white",
    legend=dict(orientation="h", y=-0.12, x=0.5, xanchor="center"),
    margin=dict(l=200),
)
fig_bar.show()

## 📈 Quantity (/90) vs Quality (Percentile) — Dual View

For each KPI we plot the **per-90 value** (x-axis = how often) against the **percentile rank** (y-axis = how good relative to peers). Top-right quadrant = high quantity AND high quality.

In [16]:
# ── Quantity vs Quality scatter plots ─────────────────────────────────────────

qty_qual_kpis = [
    ("total_passes_p90",          "total_passes_p90_pct",          "Total Passes"),
    ("forward_pass_completion_pct","forward_pass_completion_pct_pct","Fwd Pass Compl. %"),
    ("switches_p90",              "switches_p90_pct",              "Switches of Play"),
    ("long_balls_total_p90",      "long_balls_p90_pct",            "Long Balls"),
    ("forward_passes_p90",        "forward_passes_p90_pct",        "Forward Passes"),
    ("final_third_passes_p90",    "final_third_passes_p90_pct",    "Final Third Passes"),
    ("through_balls_p90",         "through_balls_p90_pct",         "Through Balls"),
    ("progressive_passes_p90",    "progressive_passes_p90_pct",    "Progressive Passes"),
    ("progressive_carries_p90",   "progressive_carries_p90_pct",   "Progressive Carries"),
    ("key_passes_p90",            "key_passes_p90_pct",            "Key Passes"),
    ("ball_recoveries_p90",       "ball_recoveries_p90_pct",       "Ball Recoveries"),
    ("tackles_won_p90",           "tackles_won_p90_pct",           "Tackles Won"),
    ("interceptions_p90",         "interceptions_p90_pct",         "Interceptions"),
    ("duels_won_pct",             "duels_won_pct_pct",             "Duels Won %"),
    ("possession_won_p90",        "possession_won_p90_pct",        "Possession Won"),
]

n = len(qty_qual_kpis)
cols = 3
rows = (n + cols - 1) // cols
fig_qq = make_subplots(rows=rows, cols=cols,
                       subplot_titles=[t[2] for t in qty_qual_kpis],
                       horizontal_spacing=0.08, vertical_spacing=0.08)

top10_names = set(ps.head(10)["player_name"])

for i, (p90_col, pct_col, label) in enumerate(qty_qual_kpis):
    r, c = i // cols + 1, i % cols + 1
    
    others = ps[~ps["player_name"].isin(top10_names)]
    fig_qq.add_trace(go.Scatter(
        x=others[p90_col], y=others[pct_col],
        mode="markers", marker=dict(size=6, color="lightgrey", opacity=0.6),
        text=others["player_name"] + " (" + others["team_name"] + ")",
        hovertemplate="%{text}<br>" + label + " p90: %{x:.2f}<br>Percentile: %{y:.0f}<extra></extra>",
        showlegend=False,
    ), row=r, col=c)
    
    top = ps[ps["player_name"].isin(top10_names)]
    fig_qq.add_trace(go.Scatter(
        x=top[p90_col], y=top[pct_col],
        mode="markers+text", marker=dict(size=10, color="#8a1f33"),
        text=top["player_name"].str.split().str[-1],
        textposition="top center", textfont=dict(size=8),
        hovertemplate="%{customdata}<br>" + label + " p90: %{x:.2f}<br>Percentile: %{y:.0f}<extra></extra>",
        customdata=top["player_name"] + " (" + top["team_name"] + ")",
        showlegend=False,
    ), row=r, col=c)
    
    fig_qq.add_hline(y=50, line_dash="dot", line_color="grey", opacity=0.4, row=r, col=c)
    fig_qq.add_vline(x=ps[p90_col].median(), line_dash="dot", line_color="grey", opacity=0.4, row=r, col=c)

fig_qq.update_layout(
    title="Quantity (p90) vs Quality (Percentile) — All KPIs<br><sub>Top 10 composite players highlighted in red | Possession-adjusted</sub>",
    height=rows * 300, template="plotly_white", showlegend=False,
)
fig_qq.update_xaxes(title_text="Per 90", title_font_size=9)
fig_qq.update_yaxes(title_text="Percentile", title_font_size=9)
fig_qq.show()

## 📋 Player Scouting Card — Quantity & Quality

Select a player from the dropdown to see their full KPI breakdown: raw per-90 value (quantity), percentile rank (quality), and pillar scores. Color-coded by percentile tier.

In [17]:
# ── Player scouting card — dropdown-based ──────────────────────────────────────

qty_qual_pairs = [
    ("total_passes_p90",          "total_passes_p90_pct",          "Total Passes",          "Build-up & Circulation"),
    ("pass_completion_pct",       "pass_completion_pct_pct",       "Pass Completion %",     "Build-up & Circulation"),
    ("backward_lateral_pct",      "backward_lateral_pct_pct",      "Back/Lateral Pass %",   "Build-up & Circulation"),
    ("forward_pass_completion_pct","forward_pass_completion_pct_pct","Fwd Pass Compl. %",    "Build-up & Circulation"),
    ("switches_p90",              "switches_p90_pct",              "Switches of Play",      "Switching Play"),
    ("long_ball_completion_pct",  "long_ball_completion_pct_pct",  "Long Ball Compl. %",    "Switching Play"),
    ("long_balls_total_p90",      "long_balls_p90_pct",            "Long Balls",            "Switching Play"),
    ("forward_passes_p90",        "forward_passes_p90_pct",        "Forward Passes",        "Progressive Passing"),
    ("final_third_passes_p90",    "final_third_passes_p90_pct",    "Final Third Passes",    "Progressive Passing"),
    ("through_balls_p90",         "through_balls_p90_pct",         "Through Balls",         "Progressive Passing"),
    ("progressive_passes_p90",    "progressive_passes_p90_pct",    "Progressive Passes",    "Progressive Passing"),
    ("progressive_carries_p90",   "progressive_carries_p90_pct",   "Progressive Carries",   "Progressive Passing"),
    ("key_passes_p90",            "key_passes_p90_pct",            "Key Passes",            "Progressive Passing"),
    ("ball_recoveries_p90",       "ball_recoveries_p90_pct",       "Ball Recoveries",       "Defensive Contribution"),
    ("tackles_won_p90",           "tackles_won_p90_pct",           "Tackles Won",           "Defensive Contribution"),
    ("interceptions_p90",         "interceptions_p90_pct",         "Interceptions",         "Defensive Contribution"),
    ("duels_won_pct",             "duels_won_pct_pct",             "Duels Won %",           "Defensive Contribution"),
    ("possession_won_p90",        "possession_won_p90_pct",        "Possession Won",        "Defensive Contribution"),
]

pillar_colors = {
    "Build-up & Circulation":  "#1f77b4",
    "Switching Play":          "#ff7f0e",
    "Progressive Passing":     "#2ca02c",
    "Defensive Contribution":  "#d62728",
}

def pctl_bg(v):
    if pd.isna(v): return "white"
    if v >= 80: return "rgba(0,180,80,0.25)"
    if v >= 60: return "rgba(0,180,80,0.10)"
    if v >= 40: return "white"
    if v >= 20: return "rgba(255,80,80,0.10)"
    return "rgba(255,80,80,0.25)"

def build_player_table(row):
    kpi_names, raw_vals, pct_vals, pillar_names = [], [], [], []
    pct_colors = []
    for raw_c, pct_c, label, pillar in qty_qual_pairs:
        kpi_names.append(label)
        rv = row.get(raw_c, np.nan)
        pv = row.get(pct_c, np.nan)
        raw_vals.append(f"{rv:.2f}" if pd.notna(rv) else "—")
        pct_vals.append(f"{pv:.0f}" if pd.notna(pv) else "—")
        pct_colors.append(pctl_bg(pv))
        pillar_names.append(pillar)

    pillar_cell_colors = [pillar_colors.get(p, "white") for p in pillar_names]
    pillar_fill = [f"rgba({int(c[1:3],16)},{int(c[3:5],16)},{int(c[5:7],16)},0.12)"
                   if c.startswith("#") else "white" for c in pillar_cell_colors]

    return go.Table(
        header=dict(
            values=["Pillar", "KPI", "Value (/90)", "Percentile"],
            fill_color="#8a1f33", font=dict(color="white", size=11),
            align="center", height=30,
        ),
        cells=dict(
            values=[pillar_names, kpi_names, raw_vals, pct_vals],
            fill_color=[pillar_fill, ["white"]*len(kpi_names), ["white"]*len(kpi_names), pct_colors],
            align=["left", "left", "center", "center"],
            font=dict(size=11), height=26,
        ),
    )

fig_card = go.Figure()
player_list_t = ps.sort_values("rank")

for i, (_, row) in enumerate(player_list_t.iterrows()):
    tbl = build_player_table(row)
    tbl.visible = True if i == 0 else False
    fig_card.add_trace(tbl)

buttons_t = []
for i, (_, row) in enumerate(player_list_t.iterrows()):
    vis = [j == i for j in range(len(player_list_t))]
    pillar_summary = (
        f"Build-up: {row['pillar_build-up_and_circulation']:.0f} | "
        f"Switch: {row['pillar_switching_play']:.0f} | "
        f"Progress: {row['pillar_progressive_passing']:.0f} | "
        f"Defend: {row['pillar_defensive_contribution']:.0f}"
    )
    buttons_t.append(dict(
        label=f"#{row['rank']} {row['player_name']} ({row['team_name']})",
        method="update",
        args=[{"visible": vis},
              {"title": f"Scouting Card — #{row['rank']} {row['player_name']} ({row['team_name']})<br>"
                        f"<sub>Composite: {row['composite_score']:.1f} | "
                        f"{row['matches']:.0f} GP, {row['total_minutes']:.0f} min | "
                        f"Poss: {row.get('avg_possession',50):.1f}% | {pillar_summary}</sub>"}],
    ))

first = player_list_t.iloc[0]
fig_card.update_layout(
    title=f"Scouting Card — #1 {first['player_name']} ({first['team_name']})<br>"
          f"<sub>Composite: {first['composite_score']:.1f} | "
          f"{first['matches']:.0f} GP, {first['total_minutes']:.0f} min</sub>",
    height=600, margin=dict(l=10, r=10, t=100, b=10),
    updatemenus=[dict(
        buttons=buttons_t, direction="down", showactive=True,
        x=0.02, xanchor="left", y=1.18, yanchor="top",
        bgcolor="white", bordercolor="#8a1f33",
    )],
)
fig_card.show()

## 🎯 Interactive Player Explorer

Select any player from the dropdown to see their full detailed radar alongside the league-average benchmark.

In [18]:
# ── Player explorer with dropdown ─────────────────────────────────────────────

radar_kpis = [
    ("total_passes_p90_pct",          "Total Passes"),
    ("pass_completion_pct_pct",       "Pass Compl. %"),
    ("backward_lateral_pct_pct",      "Back/Lateral %"),
    ("forward_pass_completion_pct_pct","Fwd Pass Compl. %"),
    ("switches_p90_pct",              "Switches"),
    ("long_ball_completion_pct_pct",  "Long Ball Compl. %"),
    ("long_balls_p90_pct",            "Long Balls"),
    ("forward_passes_p90_pct",        "Forward Passes"),
    ("final_third_passes_p90_pct",    "Final Third Passes"),
    ("through_balls_p90_pct",         "Through Balls"),
    ("progressive_passes_p90_pct",    "Progressive Passes"),
    ("progressive_carries_p90_pct",   "Progressive Carries"),
    ("key_passes_p90_pct",            "Key Passes"),
    ("ball_recoveries_p90_pct",       "Ball Recoveries"),
    ("tackles_won_p90_pct",           "Tackles Won"),
    ("interceptions_p90_pct",         "Interceptions"),
    ("duels_won_pct_pct",             "Duels Won %"),
    ("possession_won_p90_pct",        "Possession Won"),
]
radar_cols   = [c for c, _ in radar_kpis]
radar_labels = [l for _, l in radar_kpis]

fig_explore = go.Figure()

avg_vals = [50] * len(radar_labels) + [50]
fig_explore.add_trace(go.Scatterpolar(
    r=avg_vals, theta=radar_labels + [radar_labels[0]],
    line=dict(color="grey", width=1, dash="dot"), fill="none",
    name="League Average (50th pctl)", visible=True,
))

player_list = ps.sort_values("rank")

for i, (_, row) in enumerate(player_list.iterrows()):
    vals = [row.get(c, 50) for c in radar_cols]
    vals = [v if pd.notna(v) else 50 for v in vals]
    vals_closed = vals + [vals[0]]
    
    # Build hover with raw p90 + percentile
    raw_metrics = [c.replace("_pct","") for c, _ in radar_kpis]
    hover = []
    for (pct_c, lbl), raw_c in zip(radar_kpis, raw_metrics):
        raw_v = row.get(raw_c, np.nan)
        pct_v = row.get(pct_c, np.nan)
        raw_str = f"{raw_v:.2f}" if pd.notna(raw_v) else "N/A"
        pct_str = f"{pct_v:.0f}" if pd.notna(pct_v) else "N/A"
        hover.append(f"{lbl}: {raw_str} p90 | {pct_str}th pctl")
    hover.append(hover[0])
    
    fig_explore.add_trace(go.Scatterpolar(
        r=vals_closed, theta=radar_labels + [radar_labels[0]],
        fill="toself", fillcolor="rgba(138,31,51,0.12)",
        line=dict(color="#8a1f33", width=2.5),
        text=hover, hovertemplate="%{text}<extra></extra>",
        name=f"#{row['rank']} {row['player_name']}",
        visible=True if i == 0 else False,
    ))

buttons = []
for i, (_, row) in enumerate(player_list.iterrows()):
    vis = [True] + [j == i for j in range(len(player_list))]
    buttons.append(dict(
        label=f"#{row['rank']} {row['player_name']} ({row['team_name']})",
        method="update",
        args=[{"visible": vis},
              {"title": f"Player Explorer — #{row['rank']} {row['player_name']} ({row['team_name']})"
                        f"<br><sub>Composite: {row['composite_score']:.1f} | "
                        f"{row['matches']:.0f} GP, {row['total_minutes']:.0f} min | "
                        f"Poss: {row.get('avg_possession',50):.1f}% (adj: {row.get('adj_factor',1):.2f})</sub>"}],
    ))

fig_explore.update_layout(
    title="Player Explorer — Select a B2B MF to view detailed radar<br>"
          "<sub>Hover for per-90 (quantity) + percentile (quality) for each KPI</sub>",
    polar=dict(radialaxis=dict(range=[0, 100], showticklabels=True, tickfont_size=8),
               angularaxis=dict(tickfont_size=9)),
    height=700, template="plotly_white",
    updatemenus=[dict(
        buttons=buttons, direction="down", showactive=True,
        x=0.02, xanchor="left", y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#8a1f33",
    )],
)
fig_explore.show()